# CSV & Excel Parsing with LangChain

LangChain provides document loaders that convert tabular data (CSV and Excel files) into `Document` objects, which can then be chunked, embedded, and indexed for RAG pipelines just like text or PDF content.

## CSV Parsing

- **`CSVLoader`** (`langchain_community.document_loaders.CSVLoader`) reads a `.csv` file and creates **one `Document` per row**, with each column rendered as a `key: value` line in `page_content`. This row-level granularity is useful when each row represents a self-contained record (e.g., a product, a customer, an FAQ entry).
- Key parameters:
  - `file_path`: path to the CSV file.
  - `csv_args`: dict passed to Python's `csv.DictReader` (e.g., `delimiter`, `quotechar`, `fieldnames`).
  - `source_column`: which column to use as the `source` in document metadata (defaults to the file path otherwise).
  - `metadata_columns`: columns to pull out into `metadata` instead of `page_content`.
- **`UnstructuredCSVLoader`** (from `langchain_community` + the `unstructured` library) instead loads the whole CSV as a **single `Document`**, optionally rendering it as an HTML table in `metadata["text_as_html"]` — useful when you want the full table structure preserved for an LLM to reason over, rather than splitting into per-row documents.

## Excel Parsing

- **`UnstructuredExcelLoader`** handles `.xlsx` / `.xls` files via the `unstructured` library. By default it produces **one `Document` per sheet**, with `page_content` as plain text and (when `mode="elements"`) `metadata["text_as_html"]` containing an HTML rendering of the sheet's table.
- Modes:
  - `mode="single"` (default): one document per file/sheet combined.
  - `mode="elements"`: splits content into structural elements with richer metadata, useful for preserving table layout.
- Alternative approaches:
  - Load with **pandas** (`pd.read_excel` / `pd.read_csv`) and wrap rows/records manually into `Document` objects for full control over chunking and metadata.
  - **`DataFrameLoader`** converts an existing pandas `DataFrame` directly into `Document` objects, one per row, with a chosen column as `page_content` and the rest as `metadata`.

## When to Use Which

| Loader | Granularity | Best for |
|---|---|---|
| `CSVLoader` | Row-level | Structured records (FAQs, product catalogs) |
| `UnstructuredCSVLoader` | Whole file (with HTML table) | Preserving full table context |
| `UnstructuredExcelLoader` | Sheet-level | Multi-sheet workbooks, tables with formatting |
| `DataFrameLoader` | Row-level | When data is already loaded/cleaned in pandas |

Choosing row-level vs. whole-table loading is a key design decision for retrieval quality: row-level documents give precise, targeted retrieval, while whole-table documents preserve relationships between rows (e.g., totals, comparisons) at the cost of retrieval precision.


In [1]:
## PDf parsing

from pathlib import Path
import os

# Move to the project root, regardless of current notebook location
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

os.chdir(project_root)
print(f"Working directory set to: {project_root}")


Working directory set to: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [2]:
import pandas as pd

os.makedirs("data/raw/structured_files", exist_ok=True)

In [3]:
# Create sample data
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam', 'Headphones', 'Printer', 'USB Hub', 'External SSD', 'Router'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics', 'Accessories', 'Electronics', 'Accessories', 'Electronics', 'Electronics'],
    'Price': [999.99, 29.99, 79.99, 299.99, 89.99, 149.99, 199.99, 24.99, 129.99, 89.99],
    'Stock': [50, 200, 150, 75, 100, 120, 40, 300, 90, 60],
    'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation',
        'Over-ear wireless headphones with active noise cancellation',
        'All-in-one wireless printer with scanner and copier',
        '7-port USB 3.0 hub with fast data transfer',
        '1TB portable external SSD with USB-C connectivity',
        'Dual-band Wi-Fi 6 router with mesh support'
    ]
}

# Save as CSV
df = pd.DataFrame(data)
df.to_csv('data/raw/structured_files/products.csv', index=False)

In [4]:
# Save as Excel with multiple sheets
with pd.ExcelWriter('data/raw/structured_files/inventory.xlsx') as writer:
    df.to_excel(writer, sheet_name='Products', index=False)

    # Add another sheet (computed from the products data)
    summary_df = (
        df.groupby('Category')
        .agg(Total_Items=('Product', 'count'), Total_Value=('Price', 'sum'))
        .reset_index()
    )
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

## CSV Parsing

In [5]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_13708/731168459.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader
/Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
## CSV Loader - Each row becomes a document

print(" CSV Loader - Row based")

csv_loader = CSVLoader(
    file_path="data/raw/structured_files/products.csv",
    encoding="utf-8",
    csv_args={
        'delimiter':',',
        'quotechar':'"',
    }

)

csv_docs = csv_loader.load()
print(csv_docs)
print(f"Loaded {len(csv_docs)} documents (one per row)")
print("\nFirst Documents:")
print(f"Content: {csv_docs[0].page_content}")
print(f"Metadata: {csv_docs[0].metadata}")

 CSV Loader - Row based
[Document(metadata={'source': 'data/raw/structured_files/products.csv', 'row': 0}, page_content='Product: Laptop\nCategory: Electronics\nPrice: 999.99\nStock: 50\nDescription: High-performance laptop with 16GB RAM and 512GB SSD'), Document(metadata={'source': 'data/raw/structured_files/products.csv', 'row': 1}, page_content='Product: Mouse\nCategory: Accessories\nPrice: 29.99\nStock: 200\nDescription: Wireless optical mouse with ergonomic design'), Document(metadata={'source': 'data/raw/structured_files/products.csv', 'row': 2}, page_content='Product: Keyboard\nCategory: Accessories\nPrice: 79.99\nStock: 150\nDescription: Mechanical keyboard with RGB backlighting'), Document(metadata={'source': 'data/raw/structured_files/products.csv', 'row': 3}, page_content='Product: Monitor\nCategory: Electronics\nPrice: 299.99\nStock: 75\nDescription: 27-inch 4K monitor with HDR support'), Document(metadata={'source': 'data/raw/structured_files/products.csv', 'row': 4}, page

In [7]:
## Custom csv processing for better control
import pandas as pd
from typing import List
from langchain_core.documents import Document

print("\n Custom CSV processing")
def process_csv_intelligently(filepath: str) -> List[Document]:
    """Process CSV with intelligent document creation"""
    df =pd.read_csv(filepath)
    documents = []

    ## Strategy 1: One document per row wiht structured content 
    for idx, row in df.iterrows():
        # Create structured content 
        content = f""" Product information:
        Name: {row['Product']}
        Category : {row['Category']}
        Price: ${row['Price']}
        Stock: {row['Stock']} units
        Description: {row['Description']}"""

        ## Create document with rich metadata
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'row_index': idx,
                'product_name': row['Product'],
                'category': row['Category'],
                'price': row['Price'],
                'data_type': 'product_info'
            }
        )
        documents.append(doc)
    return documents

    


 Custom CSV processing


In [8]:
process_csv_intelligently("data/raw/structured_files/products.csv")

[Document(metadata={'source': 'data/raw/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'data_type': 'product_info'}, page_content=' Product information:\n        Name: Laptop\n        Category : Electronics\n        Price: $999.99\n        Stock: 50 units\n        Description: High-performance laptop with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': 'data/raw/structured_files/products.csv', 'row_index': 1, 'product_name': 'Mouse', 'category': 'Accessories', 'price': 29.99, 'data_type': 'product_info'}, page_content=' Product information:\n        Name: Mouse\n        Category : Accessories\n        Price: $29.99\n        Stock: 200 units\n        Description: Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': 'data/raw/structured_files/products.csv', 'row_index': 2, 'product_name': 'Keyboard', 'category': 'Accessories', 'price': 79.99, 'data_type': 'product_info'}, page_content=

## CSV Processing Strategies

When loading CSV data for RAG, there are two broad strategies with different tradeoffs:

### 1. Row-based (CSVLoader)

Each row becomes exactly one `Document`, with columns rendered as `key: value` pairs.

- ✅ **Simple** — one-row-one-document, minimal setup
- ✅ **Good for record lookups** — precise retrieval when a query maps to a single row (e.g., "what's the price of the Laptop?")
- ❌ **Loses table context** — no awareness of totals, comparisons, or relationships across rows (e.g., "which category has the most stock?")

### 2. Intelligent Processing

A custom pipeline that reasons over the whole table rather than row-by-row — grouping, aggregating, and enriching before chunking.

- ✅ **Preserves relationships** — keeps row-to-row and row-to-table context intact
- ✅ **Creates summaries** — pre-computed aggregates (totals, counts, category breakdowns) become retrievable facts on their own
- ✅ **Rich metadata** — each chunk can carry structured metadata (category, source row range, computed stats) for filtering
- ✅ **Better for Q&A** — handles analytical/aggregate questions, not just point lookups

### When to Choose Which

| Use case | Strategy |
|---|---|
| "Show me the details for product X" | Row-based |
| "Which category has the highest total value?" | Intelligent processing |
| Large CSVs, simple FAQ/record data | Row-based |
| Business reports, mixed analytical queries | Intelligent processing |

In practice, many production RAG pipelines combine both: row-level documents for precise lookups, plus a small number of summary documents (per category, per sheet, or per aggregate) for questions that span multiple rows.


## Excel Parsing

In [9]:
# Method 1: Using pandas for full control
print(" Pandas-based Excel Processing")
def process_excel_with_pandas(filepath: str) -> List[Document]:
    """Process Excel with sheet awareness"""
    documents = []
    
    # Read all sheets
    excel_file = pd.ExcelFile(filepath)
    
    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(filepath, sheet_name=sheet_name)
        
        # Create document for each sheet
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)
        
        doc = Document(
            page_content=sheet_content,
            metadata={
                'source': filepath,
                'sheet_name': sheet_name,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_type': 'excel_sheet'
            }
        )
        documents.append(doc)
    
    return documents

 Pandas-based Excel Processing


In [10]:
excel_docs = process_excel_with_pandas('data/raw/structured_files/inventory.xlsx')
print(f"Processed {len(excel_docs)} sheets")

Processed 2 sheets


In [11]:
excel_docs

[Document(metadata={'source': 'data/raw/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 10, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: Products\nColumns: Product, Category, Price, Stock, Description\nRows: 10\n\n     Product    Category  Price  Stock                                                 Description\n      Laptop Electronics 999.99     50         High-performance laptop with 16GB RAM and 512GB SSD\n       Mouse Accessories  29.99    200                Wireless optical mouse with ergonomic design\n    Keyboard Accessories  79.99    150                   Mechanical keyboard with RGB backlighting\n     Monitor Electronics 299.99     75                         27-inch 4K monitor with HDR support\n      Webcam Electronics  89.99    100                        1080p webcam with noise cancellation\n  Headphones Accessories 149.99    120 Over-ear wireless headphones with active noise cancellation\n     Printer Electronics 199.99     40 

In [12]:
from langchain_community.document_loaders import UnstructuredExcelLoader
# Method 2: UnstructuredExcelLoader
print("\n UnstructuredExcelLoader")
try:
    excel_loader = UnstructuredExcelLoader(
        'data/raw/structured_files/inventory.xlsx',
        mode="elements"
    )
    unstructured_docs = excel_loader.load()
    print("  ✅ Handles complex Excel features")
    print("  ✅ Preserves formatting info")
    print("  ❌ Requires unstructured library")
except Exception as e:
    print(f"Excel reading issues {e}")


 UnstructuredExcelLoader
  ✅ Handles complex Excel features
  ✅ Preserves formatting info
  ❌ Requires unstructured library


In [13]:
unstructured_docs

[Document(metadata={'source': 'data/raw/structured_files/inventory.xlsx', 'file_directory': 'data/raw/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-09-23T19:27:22', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse</td><td>Accessories</td><td>29.99</td><td>200</td><td>Wireless optical mouse with ergonomic design</td></tr><tr><td>Keyboard</td><td>Accessories</td><td>79.99</td><td>150</td><td>Mechanical keyboard with RGB backlighting</td></tr><tr><td>Monitor</td><td>Electronics</td><td>299.99</td><td>75</td><td>27-inch 4K monitor with HDR support</td></tr><tr><td>Webcam</td><td>Electronics</td><td>89.99</td><td>100</td><td>1080p webcam with noise cancellation</td></tr><tr><td>Headphones</td><td>Accessories</td><td>1